Reading data

In [2]:
import pyxdf
import pandas as pd
import os

In [3]:
from asyncio import streams
# Input XDF file
xdf_file = "../data/s7.xdf"

# Output folder
output_dir = "../data/xdf_csv_export"
os.makedirs(output_dir, exist_ok=True)
# Load File
streams, header = pyxdf.load_xdf(xdf_file)
print("Number of streams:", len(streams))

Number of streams: 3


In [4]:
for i, stream in enumerate(streams):
    print("\n==============================")
    print(f"Stream {i}")

    # Stream name (usually most useful)
    name = stream["info"]["name"][0]
    stream_type = stream["info"]["type"][0]
    channel_count = stream["info"]["channel_count"][0]

    print("Name:", name)
    print("Type:", stream_type)
    print("Channels:", channel_count)

    # Number of records (samples)
    n_samples = len(stream["time_series"])
    print("Number of records (samples):", n_samples)

    # Optional: time range
    start_time = stream["time_stamps"][0] if n_samples > 0 else None
    end_time = stream["time_stamps"][-1] if n_samples > 0 else None
    print("Time range:", start_time, "to", end_time)

    # Show a preview of first row
    if n_samples > 0:
        print("First sample:", stream["time_series"][0])


Stream 0
Name: BCIBackend
Type: BCIResult
Channels: 1
Number of records (samples): 0
Time range: None to None

Stream 1
Name: obci_eeg1
Type: EEG
Channels: 16
Number of records (samples): 0
Time range: None to None

Stream 2
Name: Unity_Markers
Type: Markers
Channels: 1
Number of records (samples): 107
Time range: 19925.88719336341 to 20083.38025279007
First sample: ['2026-06-04 17:16:01.112,BCI,TrainBCI,Eye_Closed,eye_tracking,Eye_Closed']


In [12]:
# --------------------------------------------------
# Find global reference time
# --------------------------------------------------

global_start = min(
    stream['time_stamps'][0]
    for stream in streams
)

print("Global start time:", global_start)

# --------------------------------------------------
# Export aligned streams separately
# --------------------------------------------------

for i, stream in enumerate(streams):

    # Stream metadata
    name = stream['info']['name'][0]
    stream_type = stream['info']['type'][0]

    print(f"\nProcessing stream: {name}")

    samples = stream['time_series']
    timestamps = stream['time_stamps']

    # ----------------------------------------------
    # ALIGN TIMESTAMPS
    # ----------------------------------------------

    aligned_timestamps = timestamps - global_start

    # Create dataframe
    df = pd.DataFrame(samples)

    # Add aligned timestamp column
    df.insert(0, "timestamp", aligned_timestamps)

    # Optional original timestamps
    df.insert(1, "original_timestamp", timestamps)

    # ----------------------------------------------
    # Channel names
    # ----------------------------------------------

    try:
        channels = stream['info']['desc'][0]['channels'][0]['channel']

        channel_names = [
            "timestamp",
            "original_timestamp"
        ]

        for ch in channels:
            channel_names.append(ch['label'][0])

        if len(channel_names) == len(df.columns):
            df.columns = channel_names

    except Exception:
        pass

    # ----------------------------------------------
    # Save CSV
    # ----------------------------------------------

    safe_name = name.replace(" ", "_")

    csv_path = os.path.join(
        output_dir,
        f"{i}_{safe_name}_{stream_type}.csv"
    )

    df.to_csv(csv_path, index=False)

    print(f"Saved: {csv_path}")

print("\nDone.")

Global start time: 11905.838383960649

Processing stream: obci_eeg1
Saved: ../data/xdf_csv_export/0_obci_eeg1_EEG.csv

Processing stream: BCIBackend
Saved: ../data/xdf_csv_export/1_BCIBackend_BCIResult.csv

Processing stream: Unity_Markers
Saved: ../data/xdf_csv_export/2_Unity_Markers_Markers.csv

Done.
